# Dependency Investigation

This notebook demonstrates dependencies for a publicly available item in ArcGIS Online, a StoryMap with item ID `2f9e9b3fec654af890d4b8c680167f8e`.

## Imports
We import ArcGIS and pandas tools plus the project helper function included in this project Python package.

In [2]:
from arcgis.gis import GIS
import pandas as pd

from arcgis_dependency import interrogate_item_dependencies

## Set Item ID Variable
Keep the target item ID in one variable so it is easy to reuse across the notebook.

In [3]:
storymap_item_id = "2f9e9b3fec654af890d4b8c680167f8e"

storymap_item_id

'2f9e9b3fec654af890d4b8c680167f8e'

## Instantiate GIS Object
In this case, the example is on ArcGIS Online, so we are just instantiating a GIS object with no parameters.

Ideally, replace this with a profile-based login such as `GIS(profile="your_profile")`.

In [4]:
gis = GIS()

gis

GIS @ https://www.arcgis.com version:[2026, 1]

## Retrieve Item Object
This confirms the item exists and lets you quickly inspect core metadata (title, type, owner, URL).

In [5]:
storymap_item = gis.content.get(storymap_item_id)

if storymap_item is None:
    raise ValueError(f"Item not found or inaccessible: {storymap_item_id}")

pd.DataFrame([
    {
        "id": storymap_item.id,
        "title": storymap_item.title,
        "type": storymap_item.type,
        "owner": storymap_item.owner,
        "url": getattr(storymap_item, "url", None),
    }
])

,id,title,type,owner,url
0,2f9e9b3fec654af890d4b8c680167f8e,"Big, Wild, and Connected",StoryMap,BlueWaterGIS3,https://storymaps.arcgis.com/stories/2f9e9b3fe...


## Dependency interrogation
Use helper function to recursively collect dependencies for the Web GIS Item.

### Output to Pandas

The result includes parent/dependent relationships and in-table error rows when needed.

In [6]:
dependency_df = interrogate_item_dependencies(item_ids=storymap_item_id, gis=gis)

print(f"Dependency rows returned: {len(dependency_df)}")
dependency_df.head(20)

Dependency rows returned: 48


,parent_item_id,parent_item_name,dependent_item_id,dependent_item_name
0,2f9e9b3fec654af890d4b8c680167f8e,"Big, Wild, and Connected",dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map
1,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,e97aaffef70647daa19d66d5ecdd6d8a,Current Logging Projects Rough Areas
2,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,1fdf35d8950942baa95c666f76df8423,1fdf35d8950942baa95c666f76df8423
3,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,420624cf0ad24b89bddaf8f340e62aea,2001 Roadless Areas No Wilderness
4,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,1b243539f4514b6ba35e7d995890db1d,World Hillshade
5,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,52c7896cdfab4660a595e6f6a7ef0e4d,Wilderness Areas in the United States
6,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,36502a64b0244a42baf047451f58e75d,Provinces and Territories of Canada
7,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,4e93d65f43664e43988958f5a7e2db6d,Robinson Integrated Resource Project
8,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,774019f31f8549c39b5c72f149bbe74e,USA Census States
9,dd31f9da39ff49f8848868dd020da660,Standing Trees - Current Project Map Tour Map,372c619033fa4ba1a4b716a3abaf9cd5,RevisedIRAs_GM_WM_NationalForests


## Output to Excel
Use this step when you want to share findings with teammates or archive the run.

In [7]:
from pathlib import Path

output_path = (Path.cwd() / "reports" / "storymap_2f9e9b3fec654af890d4b8c680167f8e_dependencies.xlsx").resolve()

if not output_path.parent.exists():
    output_path.parent.mkdir(parents=True, exist_ok=True)

written_path = interrogate_item_dependencies(
    item_ids=storymap_item_id,
    gis=gis,
    output_excel=output_path,
)

written_path

WindowsPath('D:/projects/arcgis-item-dependency-interrogation/notebooks/reports/storymap_2f9e9b3fec654af890d4b8c680167f8e_dependencies.xlsx')